# Course 5 Lab — Fine-Grained Authorization for Agents

**Scenario:** a procurement agent proposes a purchase order on behalf of an employee.

**Success:** the application authorizes the exact subject, agent, workload, tenant, task, action, resource, vendor, amount, country, risk result, and approval state; it blocks stale or mismatched state before any effect.

**Safety boundary:** the model proposes. Trusted application code resolves facts, authorizes, consumes limits, executes idempotently, and records observable evidence.

## Objectives and reproducibility

We will compare a role-only baseline with layered RBAC/ReBAC/ABAC/context checks, inject realistic failures, exercise the final PEP, inspect real OpenFGA SDK request objects, and compute metrics on labelled cases.

The default path is deterministic, offline, and credential-free. It imports the same `lab.py` exercised by the repository tests. No external authorization service or procurement API is contacted.

![Dual authorization combines user, task-agent, and contextual authority](assets/03-dual-authorization-context.svg)

A preview helps a UI explain a decision, but the final PEP must re-evaluate and consume state immediately before the effect. A prior allow is not a bearer capability.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from datetime import timedelta
import importlib.util
from pathlib import Path
import sys

import pandas as pd

COURSE_DIR = Path(
    "curriculum/beginner/05-fine-grained-authorization-for-agents"
).resolve()
spec = importlib.util.spec_from_file_location("course05_lab", COURSE_DIR / "lab.py")
lab = importlib.util.module_from_spec(spec)
assert spec.loader is not None
sys.modules[spec.name] = lab
spec.loader.exec_module(lab)

print("Policy:", lab.POLICY_VERSION)
print("Reference time:", lab.REFERENCE_TIME.isoformat())

## 1. Inspect the trusted contracts

`TrustedIdentityContext` comes from authenticated application state. The proposal may come from the model, but it contains no trusted role, vendor approval, risk result, or permission list. The repository resolves those facts.

In [ ]:
identity, proposal, repository, pep, adapter = lab.build_demo_environment()
pd.DataFrame(
    [
        {"source": "authenticated state", **identity.model_dump(mode="json")},
        {"source": "agent proposal", **proposal.model_dump(mode="json")},
    ]
).fillna("—")

Observe the separate subject, logical actor, workload, tenant, and task. A valid-looking identifier is not enough; each is bound again to the authoritative task grant.

## 2. Baseline — a coarse role check

The baseline checks only whether a role contains `purchase_order:create`. It cannot see the target department, vendor, amount, tenant, task, risk, or calling agent.

In [ ]:
over_scoped = lab.demo_proposal(
    amount_cents=9_999_999,
    resource_id="department:finance",
    vendor_id="vendor:unknown",
)
baseline_allowed = lab.unsafe_role_only_authorize(
    "ProcurementManager", over_scoped
)
assert baseline_allowed is True
print("Unsafe baseline allowed the forbidden request:", baseline_allowed)

This is a false permit, not merely missing detail. A broad credential may be necessary to reach an API, but the trusted PEP must impose the narrower task and business boundary.

## 3. Layered preview

The PDP resolves a task grant, resource relationship, vendor record, and proposal-bound risk assessment. The output is observable evidence—not private model reasoning.

In [ ]:
preview = repository.preview(identity, proposal, now=lab.REFERENCE_TIME)
assert preview.outcome is lab.DecisionOutcome.ALLOW
pd.DataFrame(
    {
        "field": [
            "outcome", "reason_codes", "policy_version",
            "authorization_epoch", "evidence_versions",
            "remaining_calls", "valid_until",
        ],
        "value": [
            preview.outcome.value,
            ", ".join(preview.reason_codes),
            preview.policy_version,
            preview.authorization_epoch,
            ", ".join(preview.evidence_versions),
            preview.remaining_calls,
            preview.valid_until.isoformat(),
        ],
    }
)

The request digest binds authenticated context and the exact proposal. The resolved-input digest additionally binds authoritative facts, policy version, and authorization epoch. The decision lifetime cannot outlive its supporting identity, task, or risk evidence.

## 4. Negative and boundary experiments

Each factory creates fresh authoritative state so one case cannot contaminate another. Expected denials cover independent dimensions rather than one generic “not authorized.”

In [ ]:
experiments = [
    ("wrong agent", lab.build_demo_environment(
        identity=lab.demo_identity(actor_id="agent:other"))),
    ("cross-tenant resource", lab.build_demo_environment(
        resource_tenant="tenant:other")),
    ("wrong department", lab.build_demo_environment(
        proposal=lab.demo_proposal(resource_id="department:finance"))),
    ("unapproved vendor", lab.build_demo_environment(vendor_approved=False)),
    ("task amount exceeded", lab.build_demo_environment(
        proposal=lab.demo_proposal(amount_cents=1_000_001))),
    ("risk hard limit", lab.build_demo_environment(risk_basis_points=8_000)),
]
negative_rows = []
for name, environment in experiments:
    case_identity, case_proposal, case_repository = environment[:3]
    decision = case_repository.preview(
        case_identity, case_proposal, now=lab.REFERENCE_TIME
    )
    assert decision.outcome is lab.DecisionOutcome.DENY
    negative_rows.append({
        "case": name,
        "outcome": decision.outcome.value,
        "reason_codes": ", ".join(decision.reason_codes),
    })
pd.DataFrame(negative_rows)

A hard denial is never repairable by adding a manager approval. The reasons also show why policy inputs must be application-resolved: a caller cannot make a sanctioned vendor safe by sending `approved=true`.

## 5. Escalation and bound approval

CAD 6,000 is within the task maximum but above the autonomous threshold. The first result is `ESCALATE`; a current approval bound to the full request permits final execution.

In [ ]:
approval_proposal = lab.demo_proposal(amount_cents=600_000)
(
    approval_identity,
    approval_proposal,
    approval_repository,
    approval_pep,
    approval_adapter,
) = lab.build_demo_environment(proposal=approval_proposal)

escalated = approval_repository.preview(
    approval_identity, approval_proposal, now=lab.REFERENCE_TIME
)
assert escalated.outcome is lab.DecisionOutcome.ESCALATE
receipt = lab.issue_demo_approval(
    approval_identity, approval_proposal, now=lab.REFERENCE_TIME
)
approved_result = approval_pep.execute(
    approval_identity,
    approval_proposal,
    approval=receipt,
    now=lab.REFERENCE_TIME,
)
assert approved_result.decision.outcome is lab.DecisionOutcome.ALLOW
assert approved_result.effect.status is lab.EffectStatus.APPLIED
print(escalated.outcome.value, "→", approved_result.effect.effect_id)

In [ ]:
deny_identity, deny_proposal, deny_repository, _, _ = (
    lab.build_demo_environment(
        proposal=approval_proposal,
        vendor_sanctioned=True,
    )
)
unsafe_override = lab.issue_demo_approval(
    deny_identity, deny_proposal, now=lab.REFERENCE_TIME
)
hard_deny = deny_repository.preview(
    deny_identity,
    deny_proposal,
    approval=unsafe_override,
    now=lab.REFERENCE_TIME,
)
assert hard_deny.outcome is lab.DecisionOutcome.DENY
assert "vendor_sanctioned" in hard_deny.reason_codes
print(hard_deny.reason_codes)

The receipt does not say “the request is safe.” It proves a named reviewer accepted one exact digest under one policy version and lifetime. Hard policy still applies.

## 6. TOCTOU — preview is not execution authority

We preview an allow, then revoke the vendor. Final enforcement observes the new authorization epoch and never calls the adapter.

In [ ]:
toctou_identity, toctou_proposal, toctou_repository, toctou_pep, toctou_adapter = (
    lab.build_demo_environment()
)
first_preview = toctou_repository.preview(
    toctou_identity, toctou_proposal, now=lab.REFERENCE_TIME
)
toctou_repository.replace_vendor(
    lab.VendorRecord(
        vendor_id="vendor:acme",
        tenant_id="tenant:oneplusi",
        approved=False,
        sanctioned=False,
        allowed_countries=frozenset({"CA"}),
        version="vendor-v12-revoked",
    )
)
final_result = toctou_pep.execute(
    toctou_identity, toctou_proposal, now=lab.REFERENCE_TIME
)
assert first_preview.outcome is lab.DecisionOutcome.ALLOW
assert final_result.decision.outcome is lab.DecisionOutcome.DENY
assert toctou_adapter.applied_count == 0
print(first_preview.authorization_epoch, "→", final_result.decision.authorization_epoch)
print(final_result.decision.reason_codes)

## 7. Replay, collision, and atomic call limits

The operation ID is stable across uncertain retries, while the request digest prevents reuse for mutated arguments. Usage validation and consumption share one critical section.

In [ ]:
replay_identity, replay_proposal, _, replay_pep, replay_adapter = (
    lab.build_demo_environment()
)
first = replay_pep.execute(
    replay_identity, replay_proposal, now=lab.REFERENCE_TIME
)
retry = replay_pep.execute(
    replay_identity, replay_proposal, now=lab.REFERENCE_TIME
)
collision = replay_pep.execute(
    replay_identity,
    replay_proposal.model_copy(update={"amount_cents": 400_000}),
    now=lab.REFERENCE_TIME,
)
assert first.effect == retry.effect
assert retry.decision.replayed_decision is True
assert collision.decision.reason_codes == (
    "operation_id_reused_with_different_request",
)
assert replay_adapter.applied_count == 1
print("effect count:", replay_adapter.applied_count)

In [ ]:
race_proposals = tuple(
    lab.demo_proposal(operation_id=f"OP-RACE-{index}") for index in range(8)
)
race_identity, _, _, race_pep, race_adapter = lab.build_demo_environment(
    proposal=race_proposals[0],
    additional_proposals=race_proposals[1:],
    max_calls=1,
)
with ThreadPoolExecutor(max_workers=8) as pool:
    race_results = list(
        pool.map(
            lambda item: race_pep.execute(
                race_identity, item, now=lab.REFERENCE_TIME
            ),
            race_proposals,
        )
    )
race_allows = sum(
    result.decision.outcome is lab.DecisionOutcome.ALLOW
    for result in race_results
)
assert race_allows == race_adapter.applied_count == 1
print("allows:", race_allows, "denials:", len(race_results) - race_allows)

In production, use a database transaction, compare-and-swap, or another authoritative atomic primitive. An OpenFGA condition that compares a caller-supplied count is not a transactional counter.

## 8. Outage and unknown-effect handling

A PDP outage is retryable but fails closed. An unknown external effect is different: blindly repeating it may duplicate a purchase. The operation ledger must reconcile by ID.

In [ ]:
outage_identity, outage_proposal, outage_repository, outage_pep, outage_adapter = (
    lab.build_demo_environment()
)
outage_repository.set_available(False)
unavailable = outage_pep.execute(
    outage_identity, outage_proposal, now=lab.REFERENCE_TIME
)
assert unavailable.decision.reason_codes == ("pdp_unavailable_fail_closed",)
assert outage_adapter.applied_count == 0
outage_repository.set_available(True)
recovered = outage_pep.execute(
    outage_identity, outage_proposal, now=lab.REFERENCE_TIME
)
assert recovered.effect.status is lab.EffectStatus.APPLIED
print(unavailable.decision.reason_codes, "→", recovered.effect.status.value)

In [ ]:
stale_identity, stale_proposal, stale_repository, _, _ = (
    lab.build_demo_environment()
)
stale = stale_repository.preview(
    stale_identity,
    stale_proposal,
    now=lab.REFERENCE_TIME + timedelta(minutes=21),
)
assert stale.outcome is lab.DecisionOutcome.DENY
assert {
    "identity_context_not_current",
    "task_grant_not_current",
    "risk_assessment_not_current",
}.issubset(stale.reason_codes)
print("stale evidence:", stale.reason_codes)

In [ ]:
unknown_identity, unknown_proposal, _, unknown_pep, unknown_adapter = (
    lab.build_demo_environment()
)
original_apply = unknown_adapter.apply

def lose_response(_proposal):
    raise TimeoutError("effect outcome is unknown")

unknown_adapter.apply = lose_response
try:
    unknown_pep.execute(
        unknown_identity, unknown_proposal, now=lab.REFERENCE_TIME
    )
except TimeoutError:
    pass
finally:
    unknown_adapter.apply = original_apply

uncertain_retry = unknown_pep.execute(
    unknown_identity, unknown_proposal, now=lab.REFERENCE_TIME
)
assert uncertain_retry.decision.replayed_decision is True
assert uncertain_retry.effect.status is lab.EffectStatus.UNKNOWN
assert unknown_adapter.applied_count == 0
print("retry outcome:", uncertain_retry.effect.status.value)

UNKNOWN is deliberately not converted to success or automatically re-executed. A
production reconciler would query the downstream system using the stable operation ID
before deciding whether another attempt is safe.

## Observable decision evidence

The audit event stores IDs, digests, versions, outcome, and reason codes while avoiding
raw vendor and risk payloads. In production, protect and redact this stream.

In [ ]:
audit_identity, audit_proposal, audit_repository, audit_pep, _ = (
    lab.build_demo_environment()
)
audit_result = audit_pep.execute(
    audit_identity, audit_proposal, now=lab.REFERENCE_TIME
)
audit_event = audit_repository.audit_events[0]
assert audit_event.decision_id == audit_result.decision.decision_id
assert "vendor:acme" not in audit_event.model_dump_json()
pd.DataFrame(
    {
        "field": [
            "decision_id", "request_digest", "resolved_input_digest",
            "policy_version", "authorization_epoch", "reason_codes",
        ],
        "value": [
            audit_event.decision_id,
            audit_event.request_digest,
            audit_event.resolved_input_digest,
            audit_event.policy_version,
            audit_event.authorization_epoch,
            ", ".join(audit_event.reason_codes),
        ],
    }
)

## 9. Common engine artifacts

The next cell builds actual OpenFGA Python SDK request objects without networking. Two checks are intentional: the delegating user must retain access and the task must have access while being bound to the calling agent.

In [ ]:
fga_identity, fga_proposal, _, _, _ = lab.build_demo_environment()
user_check, task_check = lab.openfga_dual_check_requests(
    fga_identity, fga_proposal
)
assert user_check.relation == "user_can_create"
assert task_check.relation == "task_can_create"
assert task_check.contextual_tuples[0].relation == "calling_agent"
print(lab.OPENFGA_MODEL)
print({"user": user_check.user, "relation": user_check.relation, "object": user_check.object})
print({"user": task_check.user, "relation": task_check.relation, "object": task_check.object, "contextual_tuples": len(task_check.contextual_tuples)})

Cedar and Rego are shown as reviewable architecture artifacts, not falsely presented as executed policies. Production Cedar should be validated against a schema and diagnostics; OPA should use a versioned bundle and a privacy-reviewed decision-log configuration.

In [ ]:
assert "forbid" in lab.CEDAR_POLICY
assert 'default decision := {"outcome": "deny"' in lab.REGO_POLICY
print("CEDAR\n", lab.CEDAR_POLICY)
print("\nREGO\n", lab.REGO_POLICY)

In [ ]:
pd.DataFrame(
    [
        {
            "option": "OpenFGA",
            "model": "relationships / tuples",
            "best fit": "user-task-agent-resource graphs",
            "production concern": "model IDs, tuple lifecycle, consistency",
        },
        {
            "option": "Cedar / Verified Permissions",
            "model": "principal-action-resource-context",
            "best fit": "analyzable application authorization",
            "production concern": "schema validation and diagnostics",
        },
        {
            "option": "OPA / Rego",
            "model": "policy over structured documents",
            "best fit": "broad service and gateway decisions",
            "production concern": "bundle rollout and decision-log privacy",
        },
        {
            "option": "Application transaction",
            "model": "typed domain checks",
            "best fit": "atomic limits and effect coordination",
            "production concern": "duplication and independent administration",
        },
    ]
)

## 10. Labelled evaluation

Counts are explicit. “Forbidden allowed” is computed only over expected denials; “false denial” only over expected allows; and “missed escalation” only over escalations.

In [ ]:
evaluation = lab.run_evaluation()
evaluation_rows = pd.DataFrame(
    [
        {
            "case": row.case,
            "expected": row.expected.value,
            "role-only baseline": row.baseline_actual.value,
            "layered": row.actual.value,
            "baseline correct": row.baseline_correct,
            "layered correct": row.correct,
            "reason": ", ".join(row.reason_codes),
        }
        for row in evaluation.rows
    ]
)
assert evaluation.baseline_correct_count == 2
assert evaluation.baseline_forbidden_allowed_count == 7
assert evaluation.baseline_missed_escalation_count == 1
assert evaluation.correct_count == evaluation.case_count == 10
assert evaluation.forbidden_allowed_count == 0
assert evaluation.false_denial_count == 0
evaluation_rows

In [ ]:
pd.DataFrame(
    [
        {
            "architecture": "role-only baseline",
            "correct / population": (
                f"{evaluation.baseline_correct_count}/{evaluation.case_count}"
            ),
            "forbidden allowed": (
                f"{evaluation.baseline_forbidden_allowed_count}/"
                f"{evaluation.forbidden_case_count}"
            ),
            "false denials": (
                f"{evaluation.baseline_false_denial_count}/"
                f"{evaluation.legitimate_case_count}"
            ),
            "missed escalations": (
                f"{evaluation.baseline_missed_escalation_count}/"
                f"{evaluation.escalation_case_count}"
            ),
        },
        {
            "architecture": "layered authorization",
            "correct / population": (
                f"{evaluation.correct_count}/{evaluation.case_count}"
            ),
            "forbidden allowed": (
                f"{evaluation.forbidden_allowed_count}/"
                f"{evaluation.forbidden_case_count}"
            ),
            "false denials": (
                f"{evaluation.false_denial_count}/"
                f"{evaluation.legitimate_case_count}"
            ),
            "missed escalations": f"0/{evaluation.escalation_case_count}",
        },
    ]
)

The ten cases prove this deterministic implementation's invariants. They are not a production benchmark. Add domain distributions, tenant/resource slices, dependency failures, revocation latency, decision latency percentiles, and verified external outcomes before making a release decision.

## Production upgrade

| Teaching component | Production replacement |
|---|---|
| dictionaries + lock | transactional database and authorization store |
| fixed identity | verified user session plus workload attestation |
| embedded version | signed policy/model deployment manifest |
| OpenFGA request objects | HA client, pinned model ID, managed tuple lifecycle |
| Cedar/Rego examples | schema/type checks, unit tests, review, staged rollout |
| in-memory adapter | durable idempotency ledger and reconciliation |
| audit list | access-controlled, redacted, integrity-protected event pipeline |

Preserve the same contract and trust boundary across these replacements.

## Exercises

1. Add currency and daily aggregate limits using authoritative totals.
2. Add optimistic versions to vendor updates and reject a stale writer.
3. Design the stored OpenFGA tuples required by `OPENFGA_MODEL`, including cleanup.
4. Add a read-only operation and justify a safe degraded mode during PDP outage.
5. Diagnose why Cedar skip-on-error makes schema validation and diagnostics important.
6. Design reconciliation for a timeout after the downstream system accepted the order.

## Summary

- Authentication and broad scopes establish reachability, not task-level authority.
- RBAC, ABAC, ReBAC, and contextual checks solve different parts of the decision.
- User authority, task-agent authority, and authoritative runtime context must all pass.
- Preview is explanatory; final authorization and atomic consumption belong at the PEP.
- Approval cannot override a hard prohibition.
- Replay, concurrency, outages, revocation, and unknown outcomes are authorization design problems—not edge cases.

Continue with the chapter's references and Course 6 for policy lifecycle and rollout.